In [4]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Tuple

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [26]:


(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Normalize data (scale pixel values between 0 and 1)
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0 

print("Train:", x_train.shape, y_train.shape)

print("Test :", x_test.shape, y_test.shape)

 

class_names = ["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"]


Train: (50000, 32, 32, 3) (50000, 1)
Test : (10000, 32, 32, 3) (10000, 1)


In [40]:
y_train_va=y_train
#vehicle=0, animal =1
#if y_train[i] is 0,1,8,9 the y_train_va[i]=0 otherwise y_train_va[i]=1
for i in range(len(y_train_va)):
    if y_train_va[i][0] in [0, 1, 8, 9]:
        y_train_va[i]=0
    else:
        y_train_va[i]=1

y_test_va=y_test
for i in range(len(y_test_va)):
    if y_test_va[i][0] in [0, 1, 8, 9]:
        y_test_va[i]=0
    else:
        y_test_va[i]=1


In [43]:

def build_multihead_cnn(input_dim:tuple[int,int,int],lr:float=1e-3)->tf.keras.Model:
    """Builds a multihead CNN model: one model predicts the class of the image, the other model predicts if it is a vehicle or animal"""
    inputs=tf.keras.Input(shape=input_dim)
    x=keras.layers.Conv2D(32, (3,3), activation='relu', padding='same')(inputs)   #image size 32x32
    x=keras.layers.MaxPooling2D((2,2))(x)                                       #image size 16x16
    x=keras.layers.Conv2D(64, (3,3), activation='relu')(x)                  
    x=keras.layers.MaxPooling2D((2,2))(x)                                       #image size 8x8
    x=keras.layers.Conv2D(128, (3,3), activation='relu')(x)
    x=keras.layers.MaxPooling2D((2,2))(x)                                       #image size 4x4
    x=keras.layers.Flatten()(x)
    x=keras.layers.Dense(128, activation='relu')(x)
    x=keras.layers.Dense(32, activation='softmax')(x)
    shared=tf.keras.layers.Dense(16, activation='relu', name="shared_repr")(x)

    class_output=tf.keras.layers.Dense(10, activation='softmax', name="class_output")(shared)
    class_vehicle_animal=tf.keras.layers.Dense(2, activation='softmax', name="class_vehicle_animal")(shared)

    model=tf.keras.Model(inputs,outputs={"class_output":class_output,"class_vehicle_animal":class_vehicle_animal})
    model.compile(
        optimizer='adam', 
        loss={"class_output":"sparse_categorical_crossentropy", "class_vehicle_animal":'sparse_categorical_crossentropy',
        },
        #Loss weights starts with 1.0/1.0
        loss_weights={"class_output":1.0, "class_vehicle_animal":1.0}, 
        metrics={
            "class_output": ['accuracy'],
            "class_vehicle_animal": ['accuracy']
        })
    return model
    
multi=build_multihead_cnn((32,32,3),lr=1e-3)
multi.summary()
    
    

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_5       │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_15 (Conv2D)  │ (None, 32, 32,    │        896 │ input_layer_5[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_15    │ (None, 16, 16,    │          0 │ conv2d_15[0][0]   │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_16 (Conv2D)  │ (None, 14, 14,    │     18,496 │ max_pooling2d_15… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_16    │ (None, 7, 7, 64)  │          0 │ conv2d_16[0][0]   │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_17 (Conv2D)  │ (None, 5, 5, 128) │     73,856 │ max_pooling2d_16… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_17    │ (None, 2, 2, 128) │          0 │ conv2d_17[0][0]   │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_5 (Flatten) │ (None, 512)       │          0 │ max_pooling2d_17… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 128)       │     65,664 │ flatten_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 32)        │      4,128 │ dense_10[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_repr (Dense) │ (None, 16)        │        528 │ dense_11[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ class_output        │ (None, 10)        │        170 │ shared_repr[0][0] │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ class_vehicle_anim… │ (None, 2)         │         34 │ shared_repr[0][0] │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 163,772 (639.73 KB)

 Trainable params: 163,772 (639.73 KB)

 Non-trainable params: 0 (0.00 B)

In [45]:
multi.fit(
    x_train,
    {
        "class_output": y_train,
        "class_vehicle_animal": y_train_va
    },
    epochs=10,
    batch_size=32,
    verbose=1,
    validation_data=(x_test, {
        "class_output": y_test,
        "class_vehicle_animal": y_test_va
    })
)



Epoch 1/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - class_output_accuracy: 1.0000 - class_output_loss: 4.8130e-04 - class_vehicle_animal_accuracy: 1.0000 - class_vehicle_animal_loss: 1.6320e-04 - loss: 6.4460e-04 - val_class_output_accuracy: 1.0000 - val_class_output_loss: 2.5901e-04 - val_class_vehicle_animal_accuracy: 1.0000 - val_class_vehicle_animal_loss: 8.7973e-05 - val_loss: 3.4698e-04
Epoch 2/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - class_output_accuracy: 1.0000 - class_output_loss: 1.6290e-04 - class_vehicle_animal_accuracy: 1.0000 - class_vehicle_animal_loss: 5.5321e-05 - loss: 2.1825e-04 - val_class_output_accuracy: 1.0000 - val_class_output_loss: 9.6079e-05 - val_class_vehicle_animal_accuracy: 1.0000 - val_class_vehicle_animal_loss: 3.2663e-05 - val_loss: 1.2874e-04
Epoch 3/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - class_output_accuracy: 1.0000 - class_output_loss: 6.2714e-05 - class_vehicle_animal_accuracy: 1.0000 - class_vehicle_animal_loss: 2.1251e-05 - 

In [ ]:
# test the multi on test data
